In [1]:
import sklearn
import numpy as np
import pandas
import torch
import sklearn.datasets
import sklearn.preprocessing
import helpers
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import OneClassSVM
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, precision_score, recall_score, precision_recall_curve, roc_auc_score, f1_score, make_scorer, auc, average_precision_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# KDDCUP Dataset
Classes:   
- 'normal' : Anomalous (1)
- Else : Non-anomalous (0)


duration: continuous.  
protocol_type: symbolic.  
service: symbolic.  
flag: symbolic.  
src_bytes: continuous.  
dst_bytes: continuous.  
land: symbolic.  
wrong_fragment: continuous.  
urgent: continuous.  
hot: continuous.  
num_failed_logins: continuous.  
logged_in: symbolic.  
num_compromised: continuous.  
root_shell: continuous.  
su_attempted: continuous.  
num_root: continuous.  
num_file_creations: continuous.  
num_shells: continuous.  
num_access_files: continuous.  
num_outbound_cmds: continuous.  
is_host_login: symbolic.  
is_guest_login: symbolic.  
count: continuous.  
srv_count: continuous.  
serror_rate: continuous.  
srv_serror_rate: continuous.  
rerror_rate: continuous.  
srv_rerror_rate: continuous.  
same_srv_rate: continuous.  
diff_srv_rate: continuous.  
srv_diff_host_rate: continuous.  
dst_host_count: continuous.  
dst_host_srv_count: continuous.  
dst_host_same_srv_rate: continuous.  
dst_host_diff_srv_rate: continuous.  
dst_host_same_src_port_rate: continuous.  
dst_host_srv_diff_host_rate: continuous.  
dst_host_serror_rate: continuous.  
dst_host_srv_serror_rate: continuous.  
dst_host_rerror_rate: continuous.  
dst_host_srv_rerror_rate: continuous.  

In [2]:
X_kddcup, Y_kddcup = helpers.load_KDD_trainset()
X_kddcup.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 494021 entries, 0 to 494020
Data columns (total 42 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   0       494021 non-null  int64  
 1   1       494021 non-null  object 
 2   2       494021 non-null  object 
 3   3       494021 non-null  object 
 4   4       494021 non-null  int64  
 5   5       494021 non-null  int64  
 6   6       494021 non-null  int64  
 7   7       494021 non-null  int64  
 8   8       494021 non-null  int64  
 9   9       494021 non-null  int64  
 10  10      494021 non-null  int64  
 11  11      494021 non-null  int64  
 12  12      494021 non-null  int64  
 13  13      494021 non-null  int64  
 14  14      494021 non-null  int64  
 15  15      494021 non-null  int64  
 16  16      494021 non-null  int64  
 17  17      494021 non-null  int64  
 18  18      494021 non-null  int64  
 19  19      494021 non-null  int64  
 20  20      494021 non-null  int64  
 21  21      49

In [3]:
Y_kddcup

0         normal.
1         normal.
2         normal.
3         normal.
4         normal.
           ...   
494016    normal.
494017    normal.
494018    normal.
494019    normal.
494020    normal.
Name: 41, Length: 494021, dtype: object

In [4]:
Y_kddcup.value_counts()

41
smurf.              280790
neptune.            107201
normal.              97278
back.                 2203
satan.                1589
ipsweep.              1247
portsweep.            1040
warezclient.          1020
teardrop.              979
pod.                   264
nmap.                  231
guess_passwd.           53
buffer_overflow.        30
land.                   21
warezmaster.            20
imap.                   12
rootkit.                10
loadmodule.              9
ftp_write.               8
multihop.                7
phf.                     4
perl.                    3
spy.                     2
Name: count, dtype: int64

Normal is treated as anomalous in this dataset

In [5]:
Y_kddcup = Y_kddcup.apply(lambda x: 1 if x=='normal.' else 0)

In [6]:
Y_kddcup.value_counts()

41
0    396743
1     97278
Name: count, dtype: int64

In [7]:
num_categorical = X_kddcup.select_dtypes(include=['object', 'category']).shape[1]
num_categorical

4

In [8]:
X_kddcup = X_kddcup.drop(X_kddcup.columns[-1], axis=1)

Need to handle nominal attributes

In [9]:
X_kddcup_onehot = pandas.get_dummies(X_kddcup,dtype=int)
X_kddcup_onehot.head()

,0,4,5,6,7,8,9,10,11,12,...,3_REJ,3_RSTO,3_RSTOS0,3_RSTR,3_S0,3_S1,3_S2,3_S3,3_SF,3_SH
0,0,181,5450,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0
1,0,239,486,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0
2,0,235,1337,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0
3,0,219,1337,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0
4,0,217,2032,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0


In [10]:
X_kddcup_onehot.info(verbose=True)
X_kddcup_onehot.columns = X_kddcup_onehot.columns.astype(str)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 494021 entries, 0 to 494020
Data columns (total 118 columns):
 #    Column         Dtype  
---   ------         -----  
 0    0              int64  
 1    4              int64  
 2    5              int64  
 3    6              int64  
 4    7              int64  
 5    8              int64  
 6    9              int64  
 7    10             int64  
 8    11             int64  
 9    12             int64  
 10   13             int64  
 11   14             int64  
 12   15             int64  
 13   16             int64  
 14   17             int64  
 15   18             int64  
 16   19             int64  
 17   20             int64  
 18   21             int64  
 19   22             int64  
 20   23             int64  
 21   24             float64
 22   25             float64
 23   26             float64
 24   27             float64
 25   28             float64
 26   29             float64
 27   30             float64
 28   31          

There is a separate test set, so we train on the full dataset instead of splitting

In [11]:
X_kddcup_train = X_kddcup_onehot[Y_kddcup==0]
Y_kddcup_anomalies = Y_kddcup[Y_kddcup==1]

In [12]:
X_kddcup_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 396743 entries, 744 to 490964
Columns: 118 entries, 0 to 3_SH
dtypes: float64(15), int64(103)
memory usage: 360.2 MB


In [13]:
def output_formatter(predictions):
    predictions = np.where(predictions == 1, 0, 1)
    return predictions

The dataset is huge, taking a subsamples to train

In [14]:
from itertools import product
sample_size = 100000
anomaly_ratio = 0.2
models = []

gammas = ['scale', 'auto', 0.001, 0.0001]
nus = [0.0001, 0.0005, 0.001, 0.01]
models = []
avg_scores = []
random_states=[41,42,43]
for (gamma, nu) in product(gammas, nus):
    model = Pipeline([
            ("scaler", StandardScaler()),
            ("svm_clf", OneClassSVM(kernel='rbf', gamma=gamma, nu=nu))
        ])
    models.append(model)
    p,r,f = 0, 0, 0
    for rs in random_states:
        shuffled_sampled = X_kddcup_train.sample(frac=1, random_state=rs).sample(n=sample_size, random_state=rs)
        anomalies = Y_kddcup_anomalies.sample(n=int(np.floor(0.2*anomaly_ratio*sample_size)), random_state=rs)
        # 80000 train, 20000 validate
        X_sample, X_test, Y_sample, Y_test = train_test_split(shuffled_sampled, Y_kddcup.iloc[shuffled_sampled.index], test_size=0.2, random_state=42)
        X_test_anomalies = pandas.concat([X_test, X_kddcup_onehot.iloc[anomalies.index]], ignore_index=True)
        Y_test_anomalies = pandas.concat([Y_test, anomalies], ignore_index=True)
        
        # Train on a shuffled sample of normal data to find promising models
        model.fit(X_sample)        

        predictions = output_formatter(model.predict(X_test_anomalies))
        p += precision_score(Y_test_anomalies, predictions) 
        r += recall_score(Y_test_anomalies, predictions)
        f += f1_score(Y_test_anomalies, predictions)
    avg_scores.append(
        [p/len(random_states), r/len(random_states), f/len(random_states)])
        

In [15]:
avg_scores = np.array(avg_scores)
avg_scores

array([[0.70295761, 0.91433333, 0.73512074],
       [0.70832395, 0.91058333, 0.73838919],
       [0.71377694, 0.91141667, 0.74098689],
       [0.94632515, 0.97108333, 0.95854269],
       [0.70153534, 0.85458333, 0.70391498],
       [0.70529074, 0.89741667, 0.73201609],
       [0.48118698, 0.89533333, 0.55448585],
       [0.71230768, 0.96808333, 0.76934476],
       [0.75190789, 0.105     , 0.18354696],
       [0.83693978, 0.10466667, 0.18543733],
       [0.85196733, 0.10475   , 0.18588743],
       [0.71975581, 0.13408333, 0.22604261],
       [0.80393741, 0.02358333, 0.04537789],
       [0.84040044, 0.024     , 0.04615155],
       [0.80317823, 0.04891667, 0.09002488],
       [0.67425467, 0.10833333, 0.18666108]])

In [16]:
indices_promising_models = np.where(avg_scores[:, 2] > 0.90)[0]
avg_scores[avg_scores[:,2] > .70]


array([[0.70295761, 0.91433333, 0.73512074],
       [0.70832395, 0.91058333, 0.73838919],
       [0.71377694, 0.91141667, 0.74098689],
       [0.94632515, 0.97108333, 0.95854269],
       [0.70153534, 0.85458333, 0.70391498],
       [0.70529074, 0.89741667, 0.73201609],
       [0.71230768, 0.96808333, 0.76934476]])

**Load Test Data**

In [17]:
X_kddcup_test, Y_kddcup_test = helpers.load_KDD_testset()

Y_kddcup_test.value_counts()

41
smurf.              164091
normal.              60593
neptune.             58001
snmpgetattack.        7741
mailbomb.             5000
guess_passwd.         4367
snmpguess.            2406
satan.                1633
warezmaster.          1602
back.                 1098
mscan.                1053
apache2.               794
processtable.          759
saint.                 736
portsweep.             354
ipsweep.               306
httptunnel.            158
pod.                    87
nmap.                   84
buffer_overflow.        22
multihop.               18
named.                  17
sendmail.               17
ps.                     16
xterm.                  13
rootkit.                13
teardrop.               12
xlock.                   9
land.                    9
xsnoop.                  4
ftp_write.               3
perl.                    2
phf.                     2
udpstorm.                2
worm.                    2
loadmodule.              2
sqlattack.               

In [18]:
# Convert labels to generalized convention
Y_kddcup_test = Y_kddcup_test.apply(lambda x: 1 if x == 'normal.' else 0)

# Drop label columns from data
X_kddcup_test = X_kddcup_test.drop(X_kddcup_test.columns[-1], axis=1)

# One-hot encode all non-continuous columns
X_kddcup_test_onehot = pandas.get_dummies(X_kddcup_test, dtype=int)
X_kddcup_test_onehot.columns = X_kddcup_test_onehot.columns.astype(str)

Y_kddcup_test.value_counts()

41
0    250436
1     60593
Name: count, dtype: int64

In [19]:
X_kddcup_test_onehot.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 311029 entries, 0 to 311028
Columns: 117 entries, 0 to 3_SH
dtypes: float64(15), int64(102)
memory usage: 277.6 MB


In [20]:
Y_kddcup_test.info()

<class 'pandas.core.series.Series'>
RangeIndex: 311029 entries, 0 to 311028
Series name: 41
Non-Null Count   Dtype
--------------   -----
311029 non-null  int64
dtypes: int64(1)
memory usage: 2.4 MB


In [21]:
X_kddcup_test_onehot.index == Y_kddcup_test.index

array([ True,  True,  True, ...,  True,  True,  True], shape=(311029,))

In [22]:
X_kddcup_test_onehot.columns

Index(['0', '4', '5', '6', '7', '8', '9', '10', '11', '12',
       ...
       '3_REJ', '3_RSTO', '3_RSTOS0', '3_RSTR', '3_S0', '3_S1', '3_S2', '3_S3',
       '3_SF', '3_SH'],
      dtype='object', length=117)

In [23]:
X_kddcup_train.columns

Index(['0', '4', '5', '6', '7', '8', '9', '10', '11', '12',
       ...
       '3_REJ', '3_RSTO', '3_RSTOS0', '3_RSTR', '3_S0', '3_S1', '3_S2', '3_S3',
       '3_SF', '3_SH'],
      dtype='object', length=118)

In [24]:
X_kddcup_test_onehot.columns == X_kddcup_train.columns

ValueError: Lengths must match to compare

**Dealing with mismatch hot encoded columns**

In [25]:
X_kddcup, _ = helpers.load_KDD_trainset()
X_kddcup = X_kddcup.drop(X_kddcup.columns[-1], axis=1)

In [26]:
X_combined = pandas.concat([X_kddcup, X_kddcup_test], axis=0)

In [27]:
X_combined_onehot = pandas.get_dummies(X_combined, dtype=int)
X_combined_onehot.columns = X_combined_onehot.columns.astype(str)

X_kddcup_train_onehot = X_combined_onehot.iloc[:len(X_kddcup), :]

# Selecting exclusively normal  data to train on
X_kddcup_train_onehot = X_kddcup_train_onehot[Y_kddcup == 0]

X_kddcup_test_onehot = X_combined_onehot.iloc[len(X_kddcup):, :]

X_kddcup_train_onehot

,0,4,5,6,7,8,9,10,11,12,...,3_REJ,3_RSTO,3_RSTOS0,3_RSTR,3_S0,3_S1,3_S2,3_S3,3_SF,3_SH
744,184,1511,2957,0,0,0,3,0,1,2,...,0,0,0,0,0,0,0,0,1,0
745,305,1735,2766,0,0,0,3,0,1,2,...,0,0,0,0,0,0,0,0,1,0
4049,79,281,1301,0,0,0,2,0,1,1,...,0,0,0,0,0,0,0,0,1,0
4113,25,269,2333,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0
7601,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
490960,0,28,0,0,3,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
490961,0,28,0,0,3,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
490962,0,28,0,0,3,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
490963,0,28,0,0,3,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


In [28]:
len(Y_kddcup_test) == len(X_kddcup_test_onehot)

True

In [29]:
scores=[]
for i in indices_promising_models:
    models[i].fit(X_kddcup_train_onehot)
    
    predictions = output_formatter(models[i].predict(X_kddcup_test_onehot))
    scores.append([precision_score(Y_kddcup_test, predictions), recall_score(Y_kddcup_test,
        predictions), f1_score(Y_kddcup_test, predictions)])

In [30]:
scores

[[0.31776923360917286, 0.9943392801148647, 0.4816224080321028]]

Clearly the best model from subsample, performed significantly worse, signalling potential subsample bias / subsample that did not retain the original dataset distribution, or the training overfitted the smaller sample.  

The randomness in subsampling I employed likely have changed the innate distribution of the bigger dataset. Should try to stratified sample instead

In [38]:
avg_scores

array([[0.70295761, 0.91433333, 0.73512074],
       [0.70832395, 0.91058333, 0.73838919],
       [0.71377694, 0.91141667, 0.74098689],
       [0.94632515, 0.97108333, 0.95854269],
       [0.70153534, 0.85458333, 0.70391498],
       [0.70529074, 0.89741667, 0.73201609],
       [0.48118698, 0.89533333, 0.55448585],
       [0.71230768, 0.96808333, 0.76934476],
       [0.75190789, 0.105     , 0.18354696],
       [0.83693978, 0.10466667, 0.18543733],
       [0.85196733, 0.10475   , 0.18588743],
       [0.71975581, 0.13408333, 0.22604261],
       [0.80393741, 0.02358333, 0.04537789],
       [0.84040044, 0.024     , 0.04615155],
       [0.80317823, 0.04891667, 0.09002488],
       [0.67425467, 0.10833333, 0.18666108]])

In [36]:
model2 = models[0].fit(X_kddcup_train_onehot)
predictions = output_formatter(model2.predict(X_kddcup_test_onehot))
scores.append([precision_score(Y_kddcup_test, predictions), recall_score(Y_kddcup_test,
                                                                            predictions), f1_score(Y_kddcup_test, predictions)])

scores[0] is the best model (obtained from subsample training)

In [37]:
scores

[[0.31776923360917286, 0.9943392801148647, 0.4816224080321028],
 [0.6284007061699335, 0.6873071146832143, 0.6565352419087856]]

In [39]:
model3 = models[1].fit(X_kddcup_train_onehot)
predictions = output_formatter(model3.predict(X_kddcup_test_onehot))
scores.append([precision_score(Y_kddcup_test, predictions), recall_score(Y_kddcup_test,
                                                                         predictions), f1_score(Y_kddcup_test, predictions)])

In [40]:
scores

[[0.31776923360917286, 0.9943392801148647, 0.4816224080321028],
 [0.6284007061699335, 0.6873071146832143, 0.6565352419087856],
 [0.6680946888559345, 0.8029970458633836, 0.7293604455070791]]

In [41]:
model4 = models[2].fit(X_kddcup_train_onehot)
predictions = output_formatter(model4.predict(X_kddcup_test_onehot))
scores.append([precision_score(Y_kddcup_test, predictions), recall_score(Y_kddcup_test,
                                                                         predictions), f1_score(Y_kddcup_test, predictions)])

In [42]:
scores

[[0.31776923360917286, 0.9943392801148647, 0.4816224080321028],
 [0.6284007061699335, 0.6873071146832143, 0.6565352419087856],
 [0.6680946888559345, 0.8029970458633836, 0.7293604455070791],
 [0.6687560005486216, 0.8046969121845757, 0.7304554953821263]]

In [43]:
model5 = models[4].fit(X_kddcup_train_onehot)
predictions = output_formatter(model5.predict(X_kddcup_test_onehot))
scores.append([precision_score(Y_kddcup_test, predictions), recall_score(Y_kddcup_test,
                                                                         predictions), f1_score(Y_kddcup_test, predictions)])

In [44]:
model6 = models[5].fit(X_kddcup_train_onehot)
predictions = output_formatter(model6.predict(X_kddcup_test_onehot))
scores.append([precision_score(Y_kddcup_test, predictions), recall_score(Y_kddcup_test,
                                                                         predictions), f1_score(Y_kddcup_test, predictions)])

In [45]:
model7 = models[7].fit(X_kddcup_train_onehot)
predictions = output_formatter(model7.predict(X_kddcup_test_onehot))
scores.append([precision_score(Y_kddcup_test, predictions), recall_score(Y_kddcup_test,
                                                                         predictions), f1_score(Y_kddcup_test, predictions)])

In [46]:
scores

[[0.31776923360917286, 0.9943392801148647, 0.4816224080321028],
 [0.6284007061699335, 0.6873071146832143, 0.6565352419087856],
 [0.6680946888559345, 0.8029970458633836, 0.7293604455070791],
 [0.6687560005486216, 0.8046969121845757, 0.7304554953821263],
 [0.23480830915677775, 0.7840674665390391, 0.3613896030792168],
 [0.27559464344491863, 0.787632234746588, 0.40831779059222123],
 [0.6835837887067395, 0.9909725545855131, 0.8090653695606772]]

Model 7 exhibit the best f1_score, keep the 7th model

In [49]:
best_model = models[7]

predictions = output_formatter(best_model.predict(X_kddcup_test_onehot))
[precision_score(Y_kddcup_test, predictions), recall_score(Y_kddcup_test, predictions), f1_score(Y_kddcup_test, predictions)]

[0.6835837887067395, 0.9909725545855131, 0.8090653695606772]

In [50]:
best_model.get_params()

{'memory': None,
 'steps': [('scaler', StandardScaler()),
  ('svm_clf', OneClassSVM(gamma='auto', nu=0.01))],
 'transform_input': None,
 'verbose': False,
 'scaler': StandardScaler(),
 'svm_clf': OneClassSVM(gamma='auto', nu=0.01),
 'scaler__copy': True,
 'scaler__with_mean': True,
 'scaler__with_std': True,
 'svm_clf__cache_size': 200,
 'svm_clf__coef0': 0.0,
 'svm_clf__degree': 3,
 'svm_clf__gamma': 'auto',
 'svm_clf__kernel': 'rbf',
 'svm_clf__max_iter': -1,
 'svm_clf__nu': 0.01,
 'svm_clf__shrinking': True,
 'svm_clf__tol': 0.001,
 'svm_clf__verbose': False}

# Isolation forest: in the context of anomaly detection

In [61]:
from sklearn.ensemble import IsolationForest


In [62]:
X_kddcup_train_onehot.info()

<class 'pandas.core.frame.DataFrame'>
Index: 396743 entries, 744 to 490964
Columns: 119 entries, 0 to 3_SH
dtypes: float64(15), int64(104)
memory usage: 363.2 MB


In [64]:
X_kddcup_test_onehot.info()

<class 'pandas.core.frame.DataFrame'>
Index: 311029 entries, 0 to 311028
Columns: 119 entries, 0 to 3_SH
dtypes: float64(15), int64(104)
memory usage: 284.8 MB


In [63]:
Y_kddcup_test.info()

<class 'pandas.core.series.Series'>
RangeIndex: 311029 entries, 0 to 311028
Series name: 41
Non-Null Count   Dtype
--------------   -----
311029 non-null  int64
dtypes: int64(1)
memory usage: 2.4 MB


In [75]:
len(Y_kddcup_anomalies) / len(X_kddcup)

0.19691065764410826

Parameters to do grid search

In [77]:
contanmination_factors = [
    float(len(Y_kddcup_anomalies) / len(X_kddcup)), 0.01, 0.001]
num_itrees = [50, 100, 150]
max_samples = [128, 'auto', 512]
max_features = [8, 10, 12, 14]

In [78]:
from itertools import product
models = []
avg_scores = []
random_states = [40, 50, 60]
for (contanmination_factor, n_tree, max_sample, max_feature) in product(contanmination_factors, num_itrees, max_samples, max_features):
    model = IsolationForest(n_estimators=n_tree, contamination=contanmination_factor,
                            max_samples=max_sample, max_features=max_feature, n_jobs=-1, random_state=42)
    p, r, f1 = 0, 0, 0
    models.append(model)
    for rs in random_states:
        model.fit(X_kddcup_train_onehot)
        predictions = output_formatter(
            model.predict(X_kddcup_test_onehot))

        p += precision_score(Y_kddcup_test, predictions)
        r += recall_score(Y_kddcup_test, predictions)
        f1 += f1_score(Y_kddcup_test, predictions)

    avg_scores.append(
        [p/len(random_states), r/len(random_states), f1/len(random_states)])

In [79]:
avg_scores = np.array(avg_scores)
avg_scores

array([[3.85058539e-01, 8.20688858e-01, 5.24178205e-01],
       [4.29556601e-01, 9.93035499e-01, 5.99701002e-01],
       [4.28371907e-01, 9.92127804e-01, 5.98380522e-01],
       [4.30683443e-01, 9.94240259e-01, 6.01018591e-01],
       [4.11241076e-01, 9.20254815e-01, 5.68453246e-01],
       [4.31215114e-01, 9.94471309e-01, 6.01578372e-01],
       [3.48775484e-01, 9.93250045e-01, 5.16266282e-01],
       [4.26496610e-01, 9.94471309e-01, 5.96971453e-01],
       [3.47669361e-01, 9.93250045e-01, 5.15053487e-01],
       [3.49105193e-01, 9.94471309e-01, 5.16792453e-01],
       [3.49886534e-01, 1.00000000e+00, 5.18393991e-01],
       [3.48903055e-01, 9.94471309e-01, 5.16570939e-01],
       [4.28234298e-01, 9.85278828e-01, 5.96995075e-01],
       [4.28962430e-01, 9.93233542e-01, 5.99157756e-01],
       [4.29689285e-01, 9.93250045e-01, 5.99869429e-01],
       [4.30577290e-01, 9.94471309e-01, 6.00957415e-01],
       [4.31563716e-01, 9.94256762e-01, 6.01878216e-01],
       [4.29916811e-01, 9.94471

In [80]:
indices = np.argsort(avg_scores[:, 2])[-3:][::-1]
indices

array([35, 16,  5])

In [82]:
scores = [avg_scores[i] for i in indices]
scores

[array([0.43099389, 1.        , 0.60236999]),
 array([0.43156372, 0.99425676, 0.60187822]),
 array([0.43121511, 0.99447131, 0.60157837])]

In [83]:
best_model = models[indices[0]]

In [84]:
best_model.get_params()

{'bootstrap': False,
 'contamination': 0.19691065764410826,
 'max_features': 14,
 'max_samples': 512,
 'n_estimators': 150,
 'n_jobs': -1,
 'random_state': 42,
 'verbose': 0,
 'warm_start': False}

Current model performances:  
    - OC-SVM( kernel="rbf", gamma='auto', nu=0.01): F1_score ~ 0.81  
    - Isolation Forest ('contamination': 0.19691065764410826, 'max_features': 14, 'max_samples': 512, 'n_estimators': 150, 'n_jobs': -1) : F1_score ~ 0.60

# Autoencoder: in the context of Anomaly Detection
- Encoder learns to 'perfectly' encode normal data, decoder learns to 'perfectly' decode normal data
- Upon receiving anomalous data, the whole system fails encode / decode said data ==> treats it as anomaly based on reconstruction error > threshold